# HE3027 Group Project: Air quality and house prices in London
## Part 1: Data preparation and exploration

This notebook loads the Land Registry pollution data, adds the crime and tube data, builds the variables
we need and describes the data. No regressions yet.

**Setup (once):** `pip install pandas numpy matplotlib jupyter`

Folder structure:

```
Code/
├── data_overview.ipynb   (this notebook)
├── data/                 (the three .dta files)
└── output/               (graphs are saved here)
```

In [3]:
%pip install pandas matplotlib numpy

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.3f}".format)

CODE_DIR = Path.cwd()          # the Code folder (where this notebook is)
DATA_DIR = CODE_DIR / "data"   # the .dta files
OUT_DIR = CODE_DIR / "output"  # graphs
OUT_DIR.mkdir(exist_ok=True)

# plot style: light grid, thin lines, one accent colour
plt.rcParams.update({
    "figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "lines.linewidth": 2, "font.size": 10,
})
ACCENT = "#2a6ebb"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load the pollution data

One row per house sale. The Stata file also stores a description (label) for each variable.

In [4]:
f_pol = DATA_DIR / "land_registry_pollution.dta"
df = pd.read_stata(f_pol, convert_categoricals=False)
with pd.read_stata(f_pol, iterator=True) as r:
    labels = r.variable_labels()

print(f"{len(df):,} sales, {df.shape[1]} variables")
pd.DataFrame({
    "label": pd.Series(labels),
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})

450,885 sales, 48 variables


,label,dtype,missing,unique
price,Sale price stated on the transfer deed,int32,0,15443
postalcode,Postalcode,str,0,66488
propertyty,"D = Detached, S = Semi-Detached, T = Terraced,...",str,0,4
age,Y= newly built ; N = established residential b...,str,0,2
duration,"F = Freehold, L= Leasehold etc. Note that Land...",str,0,2
year,Year Property was sold,int16,0,11
month,Month Property was sold,int8,0,12
day,,int8,0,31
OA01CD,,str,0,15228
OA11CD,Output Area (OA) Code 2011,str,0,15851


In [5]:
df.head()

,price,postalcode,propertyty,age,duration,year,month,day,OA01CD,OA11CD,lsoacode,msoa11,transactiondates,quarter,yearquarter,bus_distnear,tube_distnear,thamesriv_dist,Bus_stopscount_200m,percent_noedu,percent_minorities,lone_hh,detached_dum,flat_dum,semi_d_dum,terrace_dum,freehold,leasehold,heritage_count_200m,Nearest_Park_dist,postcode,oslaua,lat,lon,LA,area_code,descriptio,hectares,code_year,la_name,grossannualpay,hoursworked,jobdensity,totalpopulation,unemployment,dist_to_cbd,pm10,nox
0,320000,KT6 7NW,T,N,F,2005,10,14,E00015131,E00015131,E01003000,E02000614,2005-10-14,Q4,2005Q4,329.314,"7,126.077","6,267.038",0,0.185,0.279,0.043,0,0,0,1,1,0,0,"2,832.359",KT67NW,E09000021,51.376,-0.289,Kingston upon Thames London Borough,LAD,Local Authority District,"3,726.117",E090000212005,Kingston upon Thames,"24,937.000",37.000,0.780,152489,5.500,18.459,28.440,158.220
1,149000,KT6 6NL,F,N,L,2005,4,15,E00015078,E00015078,E01002993,E02000610,2005-04-15,Q2,2005Q2,159.703,"7,856.089","4,607.465",3,0.050,0.061,0.024,0,1,0,0,0,1,0,969.598,KT66NL,E09000021,51.388,-0.307,Kingston upon Thames London Borough,LAD,Local Authority District,"3,726.117",E090000212005,Kingston upon Thames,"24,937.000",37.000,0.780,152489,5.500,18.206,28.440,158.220
2,190000,KT6 5AB,F,N,L,2005,1,13,E00015095,E00015095,E01002997,E02000613,2005-01-13,Q1,2005Q1,21.471,"7,611.821","5,367.090",3,0.135,0.136,0.025,0,1,0,0,0,1,0,"1,770.887",KT65AB,E09000021,51.382,-0.301,Kingston upon Thames London Borough,LAD,Local Authority District,"3,726.117",E090000212005,Kingston upon Thames,"24,937.000",37.000,0.780,152489,5.500,18.407,28.440,158.220
3,227500,KT9 2EY,S,N,F,2005,5,23,E00014839,E00014839,E01002947,E02000616,2005-05-23,Q2,2005Q2,152.082,"8,325.110","7,435.170",4,0.217,0.046,0.027,0,0,1,0,1,0,0,"3,598.690",KT92EY,E09000021,51.364,-0.298,Kingston upon Thames London Borough,LAD,Local Authority District,"3,726.117",E090000212005,Kingston upon Thames,"24,937.000",37.000,0.780,152489,5.500,19.929,28.440,158.220
4,312500,KT9 1PN,S,N,F,2005,8,26,E00014814,E00014814,E01002941,E02000615,2005-08-26,Q3,2005Q3,182.275,"8,585.449","6,563.546",2,0.223,0.049,0.047,0,0,1,0,1,0,0,"2,616.463",KT91PN,E09000021,51.370,-0.308,Kingston upon Thames London Borough,LAD,Local Authority District,"3,726.117",E090000212005,Kingston upon Thames,"24,937.000",37.000,0.780,152489,5.500,19.751,28.440,158.220


## 2. Construct variables

- `lnprice`: log price, so coefficients later read as % changes
- `newbuild`: 1 if newly built
- log versions of the pollution measures
- distances in km and pay in GBP 1,000, so numbers are readable

In [6]:
df["lnprice"] = np.log(df["price"])
df["newbuild"] = (df["age"] == "Y").astype(int)
df["ln_pm10"] = np.log(df["pm10"])
df["ln_nox"] = np.log(df["nox"])
for v in ["tube_distnear", "bus_distnear", "thamesriv_dist", "Nearest_Park_dist"]:
    df[v + "_km"] = df[v] / 1000
df["pay_k"] = df["grossannualpay"] / 1000

## 3. Add the crime data

Crime counts are recorded per LSOA and year, so we merge on `lsoacode` and `year`.
Crime data only start in **2010**, so sales from 2005 to 2009 get no crime value.

In [ ]:
f_crime = DATA_DIR / "land_registry_crime.dta"
crime_cols = ["lsoacode", "year", "allcrime", "violent_crime", "propertycrime"]
crime = pd.read_stata(f_crime, columns=crime_cols, convert_categoricals=False)

# check: one value per LSOA and year
assert crime.groupby(["lsoacode", "year"])[crime_cols[2:]].nunique().max().max() == 1
crime = crime.groupby(["lsoacode", "year"], as_index=False).first()

df = df.merge(crime, on=["lsoacode", "year"], how="left")
for v in ["allcrime", "violent_crime", "propertycrime"]:
    df["log_" + v] = np.log1p(df[v])      # log(1+x), some LSOAs have 0 violent crimes

print(f"Crime years: {crime['year'].min()} to {crime['year'].max()}")
df.groupby("year")["allcrime"].apply(lambda s: s.notna().mean()).rename("share of sales with crime data")

## 4. Add the tube data

Station information is fixed per postcode, so we merge on `postalcode`.
The file is large (about 420 MB), so we only read the columns we need. The cell first checks which columns exist.

In [ ]:
f_tube = DATA_DIR / "land_registry_tube.dta"
with pd.read_stata(f_tube, iterator=True) as r:
    tube_labels = r.variable_labels()
tube_new = [c for c in tube_labels if c not in df.columns]
print("Columns in the tube file that the pollution data does not have:")
pd.Series({c: tube_labels[c] for c in tube_new}, name="label")

In [ ]:
tube_cols = [c for c in ["station_km", "stn_zone", "station", "stn_postcode"] if c in tube_labels]
tube = pd.read_stata(f_tube, columns=["postalcode"] + tube_cols, convert_categoricals=False)
tube = tube.groupby("postalcode", as_index=False).first()
df = df.merge(tube, on="postalcode", how="left")

if "stn_zone" in df:
    df["zone"] = pd.to_numeric(df["stn_zone"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
print(f"Tube data matched for {df[tube_cols[0]].notna().mean():.1%} of sales")
df[tube_cols].head()

## 5. Data checks

Things to mention in the data section of the report.

In [ ]:
check = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing %": df.isna().mean() * 100,
})
check[check["missing"] > 0].sort_values("missing", ascending=False)

- `jobdensity` is missing for all of 2015, so it can only be used on 2005 to 2014.
- Crime is missing for 2005 to 2009 by construction.

In [ ]:
print("Price percentiles (GBP):")
df["price"].quantile([0, 0.001, 0.01, 0.5, 0.99, 0.999, 1]).map("{:,.0f}".format)

In [ ]:
print("Local authorities with very few sales (outside Greater London):")
df["la_name"].value_counts().tail(5)

## 6. Descriptive statistics

In [7]:
desc_vars = ["price", "pm10", "nox", "detached_dum", "semi_d_dum", "terrace_dum", "flat_dum",
             "freehold", "newbuild", "percent_noedu", "percent_minorities", "lone_hh",
             "dist_to_cbd", "tube_distnear_km", "bus_distnear_km", "thamesriv_dist_km",
             "Nearest_Park_dist_km", "heritage_count_200m", "pay_k", "hoursworked",
             "jobdensity", "unemployment", "allcrime"] + [c for c in ["station_km", "zone"] if c in df]
desc = df[desc_vars].describe().T[["count", "mean", "std", "min", "50%", "max"]]
desc

KeyError: "['allcrime'] not in index"

In [ ]:
print("Spatial units:", {u: df[u].nunique() for u in ["la_name", "msoa11", "lsoacode", "postalcode"]})
print("Period:", df["transactiondates"].min().date(), "to", df["transactiondates"].max().date())
pd.concat([df["propertyty"].value_counts(normalize=True).rename("property type"),
           df["duration"].value_counts(normalize=True).rename("tenure"),
           df["age"].value_counts(normalize=True).rename("new build")], axis=1)

## 7. Pollution and prices over time

In [ ]:
by_year = df.groupby("year").agg(
    sales=("price", "size"),
    median_price=("price", "median"),
    mean_pm10=("pm10", "mean"),
    mean_nox=("nox", "mean"),
    monitoring_sites=("pm10", "nunique"),   # PM10 comes from the nearest site, so unique values ~ sites
)
by_year

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(by_year.index, by_year["mean_pm10"], color=ACCENT, marker="o", ms=5)
axes[0].set_title("Mean PM10 of sold homes (µg/m³)", loc="left")
axes[1].plot(by_year.index, by_year["median_price"] / 1000, color=ACCENT, marker="o", ms=5)
axes[1].set_title("Median sale price (GBP 1,000)", loc="left")
for ax in axes:
    ax.set_xlabel("Year")
fig.tight_layout()
fig.savefig(OUT_DIR / "pm10_price_by_year.png", dpi=200)

## 8. Where is pollution high?

Pollution is not random. If it is higher in places that are expensive for other reasons
(for example near the centre), a simple comparison of prices will be biased.

In [ ]:
corr_vars = ["pm10", "nox", "lnprice", "dist_to_cbd", "tube_distnear_km", "thamesriv_dist_km",
             "percent_noedu", "percent_minorities", "pay_k", "log_allcrime"]
df[corr_vars].corr().round(2)

In [ ]:
# average PM10 and log price by 1 km band of distance to the centre
band = df.assign(cbd_km=df["dist_to_cbd"].clip(upper=30).round())
by_dist = band.groupby("cbd_km").agg(pm10=("pm10", "mean"), lnprice=("lnprice", "mean"), sales=("price", "size"))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(by_dist.index, by_dist["pm10"], color=ACCENT, marker="o", ms=4)
axes[0].set_title("Mean PM10 (µg/m³)", loc="left")
axes[1].plot(by_dist.index, by_dist["lnprice"], color=ACCENT, marker="o", ms=4)
axes[1].set_title("Mean log price", loc="left")
for ax in axes:
    ax.set_xlabel("Distance to the centre (km)")
fig.tight_layout()
fig.savefig(OUT_DIR / "pm10_price_by_distance.png", dpi=200)

In [ ]:
# raw relationship: mean log price by PM10 band (no controls)
bins = df.assign(pm10_bin=pd.qcut(df["pm10"], 20))
raw = bins.groupby("pm10_bin", observed=True).agg(pm10=("pm10", "mean"), lnprice=("lnprice", "mean"))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(raw["pm10"], raw["lnprice"], color=ACCENT, marker="o", ms=6, lw=0)
ax.set_xlabel("PM10 (µg/m³), 20 equal-sized groups")
ax.set_ylabel("Mean log price")
ax.set_title("Prices and pollution without any controls", loc="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "price_vs_pm10_raw.png", dpi=200)

### How much does PM10 vary within an area?

Area fixed effects later compare sales within the same area. That only works if PM10 still varies inside
those areas. The table shows the share of PM10 variance left after removing area means
(and area-by-year means).

In [ ]:
rows = []
for unit in ["la_name", "msoa11", "lsoacode"]:
    for v in ["pm10", "nox"]:
        total = df[v].var()
        within = (df[v] - df.groupby(unit)[v].transform("mean")).var() / total
        within_year = (df[v] - df.groupby([unit, "year"])[v].transform("mean")).var() / total
        rows.append({"area": unit, "pollutant": v, "within area": within, "within area-year": within_year})
pd.DataFrame(rows).pivot(index="area", columns="pollutant").round(3)